# GeoBrain API demo

This notebook demonstrates how to: 
1. Build and save geoJSON files from Allen atlas. 
2. Compute region-level score tables
3. Render brain regions 

- Dashboard usage: `docs/dashboard_usage.md`
- Understanding scores: `docs/score_definitions.md`
- API reference: `docs/api_reference.md`

This notebook demonstrates the Python API. If you prefer an interactive workflow, the same functionality is available through the GeoBrain dashboard.

In [ ]:
import os
import json
import pandas as pd
import plotly.graph_objects as go
import brad

### 1. Build and save geoJSON files from Allen atlas. 

BRAD renders Allen atlas regions as GeoJSON polygons. In this step, we load the Allen Mouse Brain Atlas annotation volume and ontology, both of which are openly provided by the Allen Institute, and extract slices from a given orientation (e.g., coronal, sagittal, or horizontal) at different AP levels (in mm relative to Bregma). The annotation volume defines the region identity of every voxel in the atlas, while the ontology provides the hierarchical brain-region metadata used to label and annotate the resulting polygons.

Note that, in this example, we are not saving the geoJSOn file to disk. To save it, you need to call 

```python
brad.save_geojson(
    geojson_obj,
    out_path=geojson_path,
)
```

In [ ]:
volume = brad.load_annotation_volume(resolution_um=25)
structure_df = brad.load_structure_graph()
geojson_obj = brad.build_geojson(
    volume=volume,
    structure_df=structure_df,
    orientation="coronal",
    resolution_um=25,
    coords_mm=[-2.0], # Bregma -2.0 mm
    min_area_px=5,
    simplify_px=0.8,
    polygon_mode="contour",
    smooth_sigma=1.0,
)

### 2. Compute region-level score tables

Score tables summarize the distribution of detected objects across brain regions. These scores can later be visualized as choropleth maps.

The available scores and normalization methods (Within, Reference, Pooled, Group) are described in `docs/score_definitions.md`.

Note that, in this example, we are nos saving the scores to disk. To save it, you need to call: 
```python
save_scores(
    data_dir = "path/to/dir",
    out_path = "path/to/dir",
    metadata_path = "path/to/dir",
    metadata_sep = ";",
    group_col= ["group", "sex"], 
)
``` 

In [ ]:
scores = brad.score_table(
    data_dir = "/path/to/dir", 
    metadata_path= "/path/to/metadata",
    metadata_sep = ";",
    group_col= ["group", "sex"], 
)


### 3. Render brain regions

Finally, combine the GeoJSON slice and score table to generate an interactive Plotly figure.

Note that, to save the images, we need to call the function `save_geojson`: 

```python
brad.save_figure(
    fig = fig, 
    out_dir = "/path/to/dir", 
)
``` 

In [ ]:
score_df = scores["12m_female"] # we will render just one group in this case 

In [ ]:
brad.render_brain_slice(
    geojson_obj,
    score_df,
    value_col="relative_abundance_z",
)